In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

Mounted at /content/drive


In [4]:
DRIVE_PATH  = '/content/drive/MyDrive/CrispHMM/sgRNA'
BINARY_DIR  = DRIVE_PATH                             # HeLa_chrN_binary.txt files
SGRNA_CSV   = os.path.join(DRIVE_PATH, 'sgrna_features_hela.csv')
OUTPUT_DIR  = DRIVE_PATH

In [6]:
import pandas as pd
import numpy as np

BIN_SIZE = 200  # must match -b used in BinarizeBed

# Confirmed mark order from HeLa_chr1_binary.txt line 2
EXPECTED_MARKS = [
    'CTCF', 'H3K27ac', 'H3K27me3', 'H3K36me3',
    'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9ac', 'H4K20me1'
]

# Columns from sgrna_features_hela.csv to merge in
# Coordinates and raw sequence columns are excluded
SGRNA_FEATURE_COLS = [
    'gc_content',           # GC fraction of 20nt guide
    'mfe_kcal',             # ViennaRNA minimum free energy of guide secondary structure
    'tm_celsius',           # Melting temperature (Wallace rule)
    'Normalized efficacy',  # Measured CRISPR cutting efficiency (0-1)
]

In [7]:
df_sgrna = pd.read_csv(SGRNA_CSV)
print(f"Loaded {len(df_sgrna)} sgRNAs")
print(f"Columns: {df_sgrna.columns.tolist()}")
print(f"Chromosomes: {sorted(df_sgrna['Chromosome'].unique())}")

# Convert 1-based inclusive Start coordinate to 0-based bin index
# Example: Start=11736866 → (11736866 - 1) // 200 = 58684
df_sgrna['bin_index'] = (df_sgrna['Start'] - 1) // BIN_SIZE

print(f"\nSample coordinate to bin conversion:")
print(df_sgrna[['Chromosome', 'Start', 'bin_index'] + SGRNA_FEATURE_COLS].head())


Loaded 8101 sgRNAs
Columns: ['Chromosome', 'Start', 'End', 'Strand', 'sgRNA', 'Normalized efficacy', 'guide_seq', 'pam', 'gc_content', 'mfe_kcal', 'tm_celsius']
Chromosomes: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX']

Sample coordinate to bin conversion:
  Chromosome      Start  bin_index  gc_content  mfe_kcal  tm_celsius  \
0       chr4  184605984     923029        0.45      -1.9        58.0   
1       chr1   11736866      58684        0.65      -4.1        66.0   
2       chr4   25379067     126895        0.60      -2.7        64.0   
3      chr12   57936737     289683        0.45       0.0        58.0   
4       chr2  131103494     655517        0.70       0.0        68.0   

   Normalized efficacy  
0             0.177011  
1             0.001588  
2             0.498712  
3             0.114708  
4             0.250399  


In [9]:
def parse_binary_file(filepath):
    """
    Parse a ChromHMM binary .txt file.

    Returns:
        chrom    (str)        - chromosome name e.g. 'chr11'
        marks    (list)       - mark names in column order
        df_marks (DataFrame)  - binary mark matrix, one row per 200bp bin
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()

    # Line 1: "HeLa\tchrN"
    header = lines[0].strip().split('\t')
    chrom  = header[1]

    # Line 2: mark names, tab separated
    marks = lines[1].strip().split('\t')

    # Verify mark order matches expectation
    if marks != EXPECTED_MARKS:
        print(f"  Warning: mark order in {filepath} differs from expected")
        print(f"    Expected: {EXPECTED_MARKS}")
        print(f"    Got:      {marks}")

    # Lines 3+: binary values, tab separated, one row per bin
    data = []
    for line in lines[2:]:
        line = line.strip()
        if not line:
            continue
        data.append([int(x) for x in line.split('\t')])

    df_marks = pd.DataFrame(data, columns=marks)
    return chrom, marks, df_marks


In [10]:
def merge_chromosome(binary_filepath, df_sgrna):
    """
    Merge one chromosome's binary mark data with sgRNA features.

    For bins with no sgRNA target, sgRNA feature columns are NaN.
    If multiple sgRNAs fall in the same bin, the one with the highest
    Normalized efficacy is kept as representative.

    Returns merged DataFrame with one row per 200bp bin.
    """
    chrom, marks, df_marks = parse_binary_file(binary_filepath)
    n_bins = len(df_marks)
    print(f"\n  {chrom}: {n_bins:,} bins | {len(marks)} marks")

    # Add coordinate columns - kept for reference and debugging
    # bin_start and bin_end are 1-based inclusive, matching sgRNA coordinates
    df_marks.insert(0, 'chr',       chrom)
    df_marks.insert(1, 'bin_index', np.arange(n_bins, dtype=int))
    df_marks.insert(2, 'bin_start', np.arange(n_bins) * BIN_SIZE + 1)
    df_marks.insert(3, 'bin_end',   np.arange(n_bins) * BIN_SIZE + BIN_SIZE)

    # Initialise sgRNA feature columns as NaN
    for col in SGRNA_FEATURE_COLS:
        df_marks[col] = np.nan

    # Filter sgRNA table to this chromosome
    df_chr = df_sgrna[df_sgrna['Chromosome'] == chrom].copy()
    n_sgrna = len(df_chr)
    print(f"  {chrom}: {n_sgrna} sgRNAs on this chromosome")

    if n_sgrna == 0:
        print(f"  {chrom}: no sgRNAs — all feature columns will be NaN")
        return df_marks

    # Handle multiple sgRNAs in the same 200bp bin:
    # keep the one with the highest Normalized efficacy
    df_chr = (df_chr
              .sort_values('Normalized efficacy', ascending=False)
              .drop_duplicates(subset='bin_index', keep='first'))

    # Flag any sgRNAs whose bin index falls outside the chromosome length
    out_of_range = df_chr[df_chr['bin_index'] >= n_bins]
    if len(out_of_range) > 0:
        print(f"  Warning: {len(out_of_range)} sgRNAs out of range "
              f"(bin_index >= {n_bins}) - skipping:")
        print(out_of_range[['Chromosome', 'Start', 'bin_index']])
        df_chr = df_chr[df_chr['bin_index'] < n_bins]

    # Write feature values into the correct bin rows
    for _, row in df_chr.iterrows():
        idx = int(row['bin_index'])
        for col in SGRNA_FEATURE_COLS:
            df_marks.at[idx, col] = row[col]

    n_placed = df_chr['bin_index'].nunique()
    pct = n_placed / n_bins * 100
    print(f"  {chrom}: placed {n_placed} sgRNAs into bins ({pct:.3f}% coverage)")

    return df_marks

In [11]:
binary_files = sorted([
    f for f in os.listdir(BINARY_DIR)
    if f.startswith('HeLa_chr') and f.endswith('_binary.txt')
])

print(f"Found {len(binary_files)} binary files:")
for f in binary_files:
    print(f"  {f}")

summary = []

for filename in binary_files:
    filepath   = os.path.join(BINARY_DIR, filename)
    chrom_name = filename.replace('HeLa_', '').replace('_binary.txt', '')

    print(f"\nProcessing {chrom_name} ...")
    df_merged = merge_chromosome(filepath, df_sgrna)

    out_path = os.path.join(OUTPUT_DIR, f'HeLa_{chrom_name}_merged.csv')
    df_merged.to_csv(out_path, index=False)
    print(f"  Saved to {out_path}")

    summary.append({
        'chromosome': chrom_name,
        'total_bins': len(df_merged),
        'sgrna_bins': int(df_merged['Normalized efficacy'].notna().sum()),
        'pct_covered': round(
            df_merged['Normalized efficacy'].notna().sum() / len(df_merged) * 100, 4
        ),
    })

print("\n=== Run Summary ===")
print(f"{'Chromosome':<12} {'Bins':>10} {'sgRNA sites':>12} {'Coverage':>10}")
print("-" * 48)
for s in summary:
    print(f"  {s['chromosome']:<10} {s['total_bins']:>10,} "
          f"{s['sgrna_bins']:>12} {s['pct_covered']:>9.3f}%")
total_sgrna = sum(s['sgrna_bins'] for s in summary)
total_bins  = sum(s['total_bins'] for s in summary)
print("-" * 48)
print(f"  {'TOTAL':<10} {total_bins:>10,} {total_sgrna:>12} "
      f"{total_sgrna/total_bins*100:>9.3f}%")

Found 24 binary files:
  HeLa_chr10_binary.txt
  HeLa_chr11_binary.txt
  HeLa_chr12_binary.txt
  HeLa_chr13_binary.txt
  HeLa_chr14_binary.txt
  HeLa_chr15_binary.txt
  HeLa_chr16_binary.txt
  HeLa_chr17_binary.txt
  HeLa_chr18_binary.txt
  HeLa_chr19_binary.txt
  HeLa_chr1_binary.txt
  HeLa_chr20_binary.txt
  HeLa_chr21_binary.txt
  HeLa_chr22_binary.txt
  HeLa_chr2_binary.txt
  HeLa_chr3_binary.txt
  HeLa_chr4_binary.txt
  HeLa_chr5_binary.txt
  HeLa_chr6_binary.txt
  HeLa_chr7_binary.txt
  HeLa_chr8_binary.txt
  HeLa_chr9_binary.txt
  HeLa_chrM_binary.txt
  HeLa_chrX_binary.txt

Processing chr10 ...

  chr10: 677,673 bins | 9 marks
  chr10: 264 sgRNAs on this chromosome
  chr10: placed 197 sgRNAs into bins (0.029% coverage)
  Saved to /content/drive/MyDrive/CrispHMM/sgRNA/HeLa_chr10_merged.csv

Processing chr11 ...

  chr11: 675,032 bins | 9 marks
  chr11: 591 sgRNAs on this chromosome
  chr11: placed 410 sgRNAs into bins (0.061% coverage)
  Saved to /content/drive/MyDrive/CrispHMM/

In [12]:
# How many unique bins do the sgRNAs map to?
df_sgrna['bin_index'] = (df_sgrna['Start'] - 1) // 200
unique_bins = df_sgrna.groupby('Chromosome')['bin_index'].nunique().sum()
print(f"Total sgRNAs: {len(df_sgrna)}")
print(f"Unique bins:  {unique_bins}")
print(f"Duplicates:   {len(df_sgrna) - unique_bins}")

Total sgRNAs: 8101
Unique bins:  5656
Duplicates:   2445


In [13]:
sample_path = os.path.join(OUTPUT_DIR, 'HeLa_chr11_merged.csv')

if os.path.exists(sample_path):
    df_check = pd.read_csv(sample_path)

    print(f"\nchr11 merged shape: {df_check.shape}")
    print(f"Columns: {df_check.columns.tolist()}")

    print(f"\nFirst 5 rows (expect NaN in sgRNA columns):")
    print(df_check.head())

    print(f"\nFirst 3 rows WITH sgRNA data:")
    print(df_check[df_check['Normalized efficacy'].notna()].head(3))

    print(f"\nNaN counts per sgRNA column (expect ~675k NaN, ~591 real values):")
    print(df_check[SGRNA_FEATURE_COLS].isnull().sum())

    print(f"\nMark value counts (should be 0 and 1 only):")
    for mark in EXPECTED_MARKS:
        counts = df_check[mark].value_counts().to_dict()
        print(f"  {mark}: {counts}")
else:
    print("chr11 merged file not found - check OUTPUT_DIR path")


chr11 merged shape: (675032, 17)
Columns: ['chr', 'bin_index', 'bin_start', 'bin_end', 'CTCF', 'H3K27ac', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9ac', 'H4K20me1', 'gc_content', 'mfe_kcal', 'tm_celsius', 'Normalized efficacy']

First 5 rows (expect NaN in sgRNA columns):
     chr  bin_index  bin_start  bin_end  CTCF  H3K27ac  H3K27me3  H3K36me3  \
0  chr11          0          1      200     0        0         0         0   
1  chr11          1        201      400     0        0         0         0   
2  chr11          2        401      600     0        0         0         0   
3  chr11          3        601      800     0        0         0         0   
4  chr11          4        801     1000     0        0         0         0   

   H3K4me1  H3K4me2  H3K4me3  H3K9ac  H4K20me1  gc_content  mfe_kcal  \
0        0        0        0       0         0         NaN       NaN   
1        0        0        0       0         0         NaN       NaN   
2        0        0  